In [141]:
import pandas as pd
import numpy as np

data = pd.read_csv('quarterback_stats.csv')
ids = [33106, 34869, 36264, 39918]
names = ['Jared Goff', 'Sam Darnold', 'Jordan Love', 'Caleb Williams']


def find_negative_stats(df: pd.DataFrame):

    negative_values = ['sack', 'interception', 'qb_hit', 'fumble']
    colnames = []
    for val in negative_values:
        colnames.extend([colname for colname in df.columns if val in colname])
    return colnames


qualified = (
    data
    .loc[(data['games'] >= 4) & (data['attempts'] >= 150)]
)
qualified_numeric = qualified.drop(columns='player_id') # drop id column before standardization
# nomalize data
qualified_norm = (
    qualified_numeric - qualified_numeric.mean()
) / (
    qualified_numeric.std()
)
qualified_norm['player_id'] = qualified['player_id'].copy()

negative_columns = find_negative_stats(qualified_norm)

qualified_norm_melted = qualified_norm.melt(
    id_vars='player_id', var_name='stat', value_name='norm_value'
)
# reverse values for negative stats
negative_column_mask = (
    qualified_norm_melted
    ['stat']
    .apply(lambda x: x in negative_columns)
    # go from 0 for postive stat 1 for negative stat
    # to -1 for negative stat 1 for positive stat
    * 2 # expand
    * -1 # shift
    + 1 # flip
)

qualified_norm_melted['norm_value'] = qualified_norm_melted['norm_value'] * negative_column_mask

def weighted_random(df:pd.DataFrame):
    # df is a dataframe with columns stat and norm_value
    stat_mask = (df['stat'].str.contains('per') - 1) * -1 + 1 # weight non 'per' columns 2 times as heavy
    values = np.array(df['norm_value'] * (stat_mask))
    weighted_dist = values / values.sum()
    return np.random.choice(a=df['stat'].to_numpy(), size=int(len(df['stat']) / 2), p=weighted_dist, replace=False)


# find worst stats for best player
best_players_worst_stats = (
    qualified_norm_melted
    .loc[
        qualified_norm_melted['player_id'] == ids[0]
    ]
    .sort_values('norm_value')
    .iloc[:10]
)
worst_players_best_stats = (
    qualified_norm_melted
    .loc[
        qualified_norm_melted['player_id'] == ids[-1]
    ]
    .sort_values('norm_value', ascending=False)
    .iloc[:10]
)

#best_players_worst_stats['stat']
stats = list(weighted_random(best_players_worst_stats)) + list(weighted_random(worst_players_best_stats))

# find all ranks for these stats
# using melted becuase negative ranks will be accounted for
stat_ranks = (
    qualified_norm_melted
    .query('stat in @stats')
    .pivot(
        columns='stat',
        values='norm_value', 
        index='player_id'
    )
    .rank(ascending=False, method='first')
    .reset_index()
    .query('player_id in @ids')
    .melt(
        id_vars='player_id',
        value_name='rank',
        var_name='stat'
    )
)

stat_values = (
    qualified
    .loc[
        qualified['player_id'].isin(ids), ['player_id'] + stats
    ]
    # melt back down to long format
    .melt(
        id_vars='player_id', 
        var_name='stat',
        value_name='value'
    )
)

supporting_stats = (
    pd.merge(
        stat_ranks, stat_values,
        how='inner',
        on=['player_id', 'stat']
    )
)
# add names
supporting_stats['name'] = supporting_stats['player_id'].map(dict(zip(ids, names)))

tables = []

for stat in supporting_stats['stat'].unique():
    md = supporting_stats.loc[supporting_stats['stat'] == stat, ['name', 'rank', 'value']].to_markdown(index=False)
    tables.append((stat, md))

for stat, table in tables:
    print(stat)
    print()
    print(table)

air_yards_per_game_third_or_fourth_down

| name           |   rank |    value |
|:---------------|-------:|---------:|
| Jared Goff     |     37 |  60.1111 |
| Jared Goff     |     37 |  60.1111 |
| Sam Darnold    |      4 |  94.6111 |
| Sam Darnold    |      4 |  94.6111 |
| Jordan Love    |     10 |  91      |
| Jordan Love    |     10 |  91      |
| Caleb Williams |      2 | 106.059  |
| Caleb Williams |      2 | 106.059  |
air_yards_per_pass_attempt_third_down

| name           |   rank |    value |
|:---------------|-------:|---------:|
| Jared Goff     |     38 |  6.52667 |
| Sam Darnold    |     14 |  8.63889 |
| Jordan Love    |      4 | 10.5794  |
| Caleb Williams |     19 |  8.36364 |
attempts

| name           |   rank |   value |
|:---------------|-------:|--------:|
| Jared Goff     |     10 |     539 |
| Sam Darnold    |      8 |     545 |
| Jordan Love    |     19 |     425 |
| Caleb Williams |      7 |     562 |
interceptions_per_game

| name           |   rank |    val